---
title: "潜在因子模型"
date: "2025-07-20"
excerpt: "双向固定效应（TWFE）是实证研究的'标准动作'，但在处理效应异质性和非平行趋势面前，它正面临严峻挑战。本文总结了从传统 DiD 到潜在因子模型（LFM）的进阶路径。"
tags: ["双重差分", "固定效应", "因果推断", "潜在因子模型"]
category: ["经济学","econometrics"]
language: "zh"
---


## 放松平行趋势

前面介绍了对于处理效应异质性稳健的估计方法。现在我们进一步放宽假设，如果平行趋势不成立呢？

如果平行趋势不成立，我们不应寻找"匹配者"，而应 **"合成"或"预测"反事实**。

| 方法 | 适用场景 | 核心思想 |
|------|----------|----------|
| **合成控制法 (SCM)** | 单一处理单位 | 通过加权控制组来模拟处理组 |
| **潜在因子模型 (LFM)** | 多单位、大 T 大 N | 用因子结构预测反事实 |

> **思考：PSM-DiD 是否可行？**
> 
> 答案是：**需谨慎！** PSM 解决的是基于可观测变量的选择性偏差（Selection on Observables），本质上是匹配而非合成。它依然要求满足平行趋势假设，且在小样本下容易引入额外偏误。

## 3. 核心前沿：基于潜在因子模型的反事实估计

根据徐轶青教授的总结，潜在因子模型将因果推断视为**缺失数据问题**。

### 3.1 核心思想

> 在面板数据中，将 $Y(1)$ 当作缺失数据，基于结果模型预测 $Y(0)$

其核心逻辑是引入**交互式固定效应（Interactive Fixed Effects）**：

$$Y_{it}(0) = X'_{it}\beta + \alpha_i + \xi_t + \lambda'_i f_t + \varepsilon_{it}$$

### 3.2 交叉项到底是什么？

很多人第一次看到 $\lambda'_i f_t$ 都会困惑：这到底是什么？

**通俗理解**：
- $f_t$ 是"公共因子"——比如宏观经济冲击、技术变革、政策环境等**随时间变化的共同力量**
- $\lambda_i$ 是"因子载荷"——每个个体对上述共同力量的**敏感程度/反应强度**

**举个例子**：
> 假设 $f_t$ 是"经济周期"，那么 $\lambda_i$ 就表示"某个地区对经济周期的敏感度"。
> 沿海地区可能 $\lambda_i$ 很大（出口导向，对全球经济敏感），内陆地区可能 $\lambda_i$ 较小。

### 3.3 为什么交叉项能解决平行趋势问题？

传统 TWFE 假设所有个体对时间趋势的反应是**同质**的（都只有 $\xi_t$）。

但现实中，不同个体对宏观趋势的反应是**异质**的：

$$\text{传统 TWFE: } Y_{it} = \alpha_i + \xi_t + \beta D_{it} + \varepsilon_{it}$$

$$\text{IFE 模型: } Y_{it} = \alpha_i + \xi_t + \lambda'_i f_t + \beta D_{it} + \varepsilon_{it}$$

加入 $\lambda'_i f_t$ 后，每个个体可以有自己的"时间趋势"，从而**放松了平行趋势假设**。

从图形上来理解，即为，使用0处的数据估计1，2，3，4，5，-1，-2，-3，-4，-5

![图示潜在因子](images/posts/潜在因子.png)

## 4. 主要估计方法

### 4.1 IFEct（交互式固定效应反事实估计）

**代表文献**：Xu (2017), Gobillon & Magnac (2016)

**估计步骤**（三步法）：

1. **训练模型**：仅使用控制组（$D_{it}=0$）的观测值估计 $\lambda_i$ 和 $f_t$
2. **预测反事实**：用估计出的参数预测处理组的 $Y_{it}(0)$
3. **计算 ATT**：$\hat{\tau}_{it} = Y_{it} - \hat{Y}_{it}(0)$

**技术细节**：通过迭代算法（如主成分分析 PCA）同时估计公共因子和个体载荷。

### 4.2 矩阵补全法（Matrix Completion）

**代表文献**：Athey et al. (2021)

**核心思想**：
> 把面板数据看作一个矩阵，其中处理后的观测值是"缺失"的。利用矩阵的低秩结构来"填补"这些缺失值。

**数学表述**：

$$\min_{L} \sum_{(i,t): D_{it}=0} (Y_{it} - L_{it})^2 + \lambda ||L||_*$$

其中 $||L||_*$ 是**核范数**（所有奇异值之和），用于正则化，保证低秩解。

### 4.3 IFEct vs. MC：正则化方式的差异

| 方法 | 正则化方式 | 特点 |
|------|------------|------|
| IFEct | 固定因子维度 $r$ | 需要预先指定因子个数 |
| MC (best subset) | 子集选择 | 离散优化，计算复杂 |
| MC (nuclear norm) | 核范数松弛 | 凸优化，计算高效 |

**奇异值收缩公式**：

$$\sigma_k^* = (\sigma_k - \lambda)_+ = \max(\sigma_k - \lambda, 0)$$

## 5. 推断与诊断

### 5.1 非参 Block Bootstrap

```python
# 伪代码：Block Bootstrap 流程
for b in 1:B:
    # 以个体为单位有放回抽样
    sample_units = bootstrap_sample(units, replace=True)
    # 重新估计模型
    tau_b = estimate_ife(sample_units)
# 用 B 次 bootstrap 估计量的方差作为标准误
se = std(tau_1, ..., tau_B)
```

- 适用于大 $T$ 和大 $N$ 场景
- 保持时间序列依赖结构

### 5.2 基于排列的检验（Permutation Test）

**Sharp Null 假设**下的排列检验（Chernozhukov et al., 2019）：

- 跨时间（按块）随机化，而不是跨单元
- 如果 $T$ 很大，误差项平稳弱依赖，则有效

### 5.3 诊断测试

Liu, Wang & Xu (2022) 提出了一套完整的诊断测试方法：

- 因子维度选择检验
- 模型设定检验
- 残差诊断

## 6. 实证应用示例

### Fouirnaies & Mutlu-Eren (2015)：党派结盟与特别资助

| 项目 | 内容 |
|------|------|
| **研究问题** | 地方与中央政府的党派结盟是否会带来更多特别资助？ |
| **分析对象** | 1992-2012 年间的 466 个英国地方议会 |
| **处理组** | 地方和中央政府之间存在党派结盟 |
| **结果变量** | 特别资助数额（special grant） |

**稳健性检验**：
- 去掉 3 期数据，检验 Carryover Effects
- 安慰剂检验：随机化处理时间

## 7. 实证建议

### 7.1 数据可视化

**必须绘制处理状态图（Treatment Status Plot）**：

```python
import seaborn as sns
import matplotlib.pyplot as plt

# 热力图展示处理状态
pivot_data = data.pivot(index='unit', columns='year', values='treated')
sns.heatmap(pivot_data, cmap='YlOrRd', cbar_kws={'label': 'Treated'})
plt.xlabel('Year')
plt.ylabel('Unit')
plt.title('Treatment Status Over Time')
plt.show()
```

清楚地看到 $N$ 和 $T$ 的分布是选择方法的前提。

### 7.2 明确实验设计

思考处理状态是如何分配的？问自己：
> **如果这是一个理想化的随机对照实验，它长什么样？**

### 7.3 从基准开始

```
如果反馈效应弱 → 先跑 DiD / FEct
        ↓
预趋势检验未通过？
        ↓
果断转向 IFEct / augsynth / 矩阵补全
```

### 7.4 稳健性检验清单

| 检验类型 | 具体操作 |
|----------|----------|
| **安慰剂检验** | 改变处理时间点、随机化处理组 |
| **排除混杂因素** | 结合 NLP 技术（如 BERT）从文本中提取控制变量 |
| **Carryover 检验** | 剔除处理后若干期，观察结果稳定性 |

### 7.5 推断的安全性

当 $N_{tr}$（处理单位）较小时：

- **聚类 Bootstrap** 比传统聚类标准误更可靠
- **Jackknife** 可以作为稳健性检查

## 8. 方法选择流程图

```mermaid
graph TD
    A[面板数据因果推断] --> B{平行趋势假设成立？}
    B -->|是 | C{处理效应同质？}
    B -->|否 | D[IFEct / 矩阵补全]
    C -->|是 | E[TWFE / DiD]
    C -->|否 | F[Sun & Abraham / Callaway & Sant'Anna]
    D --> G{大 T 大 N?}
    G -->|是 | H[IFEct 或 MC]
    G -->|否 | I[传统 SCM]
```

### 重点提要

| 结论 | 说明 |
|------|------|
| DiD 是 FEct 的特例 | 当因子维度为 0 时，FEct 退化为 TWFE |
| FEct = BJS 插补估计 | 计算方式等价 |
| IFEct 和 MC 用迭代算法 | 需要交叉验证选择调整参数 |
| 不允许负权重 | SCM 类方法的限制 |
| 需要大 T 大 N | 因子模型的估计要求 |

## 9. 总结与反思

### 9.1 方法局限性

没有任何一种方法是万能钥匙。潜在因子模型的局限包括：

1. **数据门槛高**：需要大 $T$ 和大 $N$ 的面板数据
2. **模型依赖性强**：因子维度选择、正则化参数都影响结果
3. **计算成本高**：迭代算法比 OLS 慢得多

### 9.2 写给申请季的你

对于像 John Klopfer 这样重视行政数据和实证细节的导师，展示你对**模型局限性**的深刻理解，比单纯列出显著的 $\beta$ 更具说服力：

> - 主动讨论负权重问题的来源与解决方案
> - 解释因子结构选择的依据（特征值 scree plot、交叉验证等）
> - 展示多种估计方法的结果对比
> - 诚实报告安慰剂检验的"失败"案例

### 9.3 一句话总结

> **合成控制法和基于潜在因子模型的反事实估计量，为我们提供了一把放松平行趋势假设的钥匙——但钥匙能否打开锁，还要看数据质量和模型设定是否匹配。**

## 参考文献

1. De Chaisemartin, C., & D'Haultfoeuille, X. (2020). Two-Way Fixed Effects Estimators with Heterogeneous Treatment Effects. *American Economic Review*, 110(9), 2964-2996.

2. Sun, L., & Abraham, S. (2021). Estimating Dynamic Treatment Effects in Event Studies with Heterogeneous Treatment Effects. *Journal of Econometrics*, 225(2), 175-199.

3. Callaway, B., & Sant'Anna, P. H. (2021). Difference-in-Differences with Multiple Time Periods. *Journal of Econometrics*, 225(2), 200-230.

4. Xu, Y. (2017). Generalized Synthetic Control Method: Causal Inference with Interactive Fixed Effects Models. *Political Analysis*, 25(1), 57-76.

5. Gobillon, L., & Magnac, T. (2016). Regional Policy Evaluation: Interactive Fixed Effects and Synthetic Controls. *Review of Economics and Statistics*, 98(3), 535-551.

6. Athey, S., Bayati, M., Doudchenko, N., Imbens, G., & Khosravi, K. (2021). Matrix Completion Methods for Causal Panel Data Models. *Journal of the American Statistical Association*, 116(536), 1716-1730.

7. Liu, L., Wang, Y., & Xu, Y. (2021). A Practical Guide to Counterfactual Estimators for Causal Inference with Time-Series Cross-Sectional Data. *American Journal of Political Science*.

8. Arkhangelsky, D., Athey, S., Hirshberg, D. A., Imbens, G. W., & Wager, S. (2021). Synthetic Difference In Differences. *American Economic Review*, 111(12), 4088-4118.